# Pretraining — Advanced
> Section 01 — Topic 05 — advanced

**Prerequisites:** [05-pretraining/foundations.ipynb](./foundations.ipynb)

**What you'll learn:**
- Distributed training with DDP, DeepSpeed, FSDP
- Mixed precision and gradient accumulation at scale

**What you'll build:**
- Multi-GPU training configurations

## Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import Dataset, DataLoader, DistributedSampler
import matplotlib.pyplot as plt
import numpy as np
import math
import os
import json
import tempfile
from typing import Optional

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU count: {torch.cuda.device_count()}")
    print(f"GPU name: {torch.cuda.get_device_name(0)}")

## 1. Distributed Training Fundamentals — DDP

A single GPU simply cannot hold or efficiently train modern language models. Even a relatively small 1B parameter model requires ~4 GB just for the parameters in fp32, plus ~12 GB for optimizer states (AdamW stores two momentum buffers), plus memory for activations and gradients. Training such a model requires distributed strategies.

Distributed Data Parallel (DDP) is the simplest and most widely used approach. The core idea is straightforward: replicate the model on every GPU, give each GPU a different subset of the data, and synchronize gradients across GPUs after each backward pass. This is mathematically equivalent to training on a single GPU with a larger batch size — the gradient from each GPU is averaged via an all-reduce operation, and then every GPU applies the same update.

DDP has minimal communication overhead compared to more advanced strategies because it only communicates gradients (not model parameters or optimizer states), and the all-reduce can overlap with the backward pass using bucketed gradient reduction. However, DDP requires that the full model fits on each GPU, which limits the model size you can train.

The key abstraction in PyTorch DDP is process groups. Each GPU runs a separate process, identified by its rank (0, 1, 2, ...). The processes communicate via NCCL (NVIDIA Collective Communications Library) on GPU clusters. The world size is the total number of processes.

In [ ]:
# A minimal transformer for demonstrations
class MiniTransformerLM(nn.Module):
    def __init__(self, vocab_size=10000, d_model=256, n_heads=4, n_layers=4, max_seq_len=512):
        super().__init__()
        self.d_model = d_model
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 4,
            dropout=0.1, activation="gelu", batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight  # Weight tying
    
    def forward(self, input_ids):
        B, T = input_ids.shape
        positions = torch.arange(T, device=input_ids.device).unsqueeze(0)
        x = self.tok_emb(input_ids) + self.pos_emb(positions)
        causal_mask = nn.Transformer.generate_square_subsequent_mask(T, device=input_ids.device)
        x = self.transformer(x, mask=causal_mask, is_causal=True)
        x = self.ln_f(x)
        return self.lm_head(x)


def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

model = MiniTransformerLM()
total, trainable = count_parameters(model)
print(f"Total parameters: {total:,}")
print(f"Trainable parameters: {trainable:,}")
print(f"Model size (fp32): {total * 4 / 1024**2:.1f} MB")

In [ ]:
def write_ddp_training_script():
    """
    Write a complete DDP training script to disk.
    DDP must be launched as separate processes (one per GPU),
    so it cannot run inside a notebook cell directly.
    
    Launch with: torchrun --nproc_per_node=NUM_GPUS train_ddp.py
    """
    script = '''
"""Distributed Data Parallel (DDP) training script.

Launch with:
    torchrun --nproc_per_node=NUM_GPUS train_ddp.py
"""
import os
import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import Dataset, DataLoader, DistributedSampler
import math


def setup_distributed():
    """Initialize the distributed process group."""
    # torchrun sets these environment variables automatically
    dist.init_process_group(backend="nccl")
    local_rank = int(os.environ["LOCAL_RANK"])
    torch.cuda.set_device(local_rank)
    return local_rank


def cleanup_distributed():
    dist.destroy_process_group()


class RandomTokenDataset(Dataset):
    """Simple dataset for demonstration."""
    def __init__(self, num_samples, seq_len, vocab_size):
        self.data = torch.randint(0, vocab_size, (num_samples, seq_len))
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return self.data[idx]


class SimpleLM(nn.Module):
    def __init__(self, vocab_size=10000, d_model=256, n_heads=4, n_layers=4, max_seq_len=256):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 4,
            dropout=0.1, activation="gelu", batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight

    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device).unsqueeze(0)
        h = self.tok_emb(x) + self.pos_emb(pos)
        mask = nn.Transformer.generate_square_subsequent_mask(T, device=x.device)
        h = self.transformer(h, mask=mask, is_causal=True)
        h = self.ln_f(h)
        return self.lm_head(h)


def train():
    local_rank = setup_distributed()
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    device = torch.device(f"cuda:{local_rank}")

    # Create model on this GPU
    model = SimpleLM().to(device)
    # Wrap with DDP — this handles gradient synchronization
    model = DDP(model, device_ids=[local_rank])

    # Dataset + distributed sampler
    dataset = RandomTokenDataset(num_samples=10000, seq_len=256, vocab_size=10000)
    # DistributedSampler ensures each GPU gets different data
    sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank, shuffle=True)
    dataloader = DataLoader(dataset, batch_size=8, sampler=sampler, drop_last=True)

    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)

    # Training loop
    model.train()
    for epoch in range(2):
        sampler.set_epoch(epoch)  # Important: shuffle differently each epoch
        for step, batch in enumerate(dataloader):
            batch = batch.to(device)
            logits = model(batch)
            # Shifted CLM loss
            loss = nn.functional.cross_entropy(
                logits[:, :-1].reshape(-1, logits.size(-1)),
                batch[:, 1:].reshape(-1)
            )
            loss.backward()  # DDP automatically syncs gradients here
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()

            if rank == 0 and step % 50 == 0:
                print(f"Epoch {epoch} Step {step} Loss: {loss.item():.4f}")

    # Save model (only rank 0)
    if rank == 0:
        torch.save(model.module.state_dict(), "model_ddp.pt")
        print("Model saved.")

    cleanup_distributed()


if __name__ == "__main__":
    train()
'''
    return script


ddp_script = write_ddp_training_script()
print("DDP Training Script:")
print("=" * 60)
print(ddp_script)
print("=" * 60)
print("\nLaunch with: torchrun --nproc_per_node=NUM_GPUS train_ddp.py")

### How All-Reduce Works

The all-reduce operation is the core communication primitive in DDP. After each backward pass, every GPU has computed gradients for its local data shard. The all-reduce operation computes the average gradient across all GPUs and distributes the result to every GPU. This is mathematically equivalent to computing the gradient over the union of all data shards.

The ring all-reduce algorithm is the most common implementation. In a ring of N GPUs, each GPU sends a chunk of its gradients to the next GPU in the ring while simultaneously receiving a chunk from the previous GPU. After 2(N-1) steps, every GPU has the complete averaged gradient. The total communication volume per GPU is 2(N-1)/N times the gradient size, which approaches 2x the gradient size for large N — this means the communication cost is nearly independent of the number of GPUs.

PyTorch DDP uses bucketed all-reduce: instead of waiting for the entire backward pass to complete, it groups gradients into buckets and starts the all-reduce for each bucket as soon as all its gradients are computed. This overlaps communication with computation, significantly reducing the effective communication overhead.

In [ ]:
# Visualize all-reduce communication pattern
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Ring all-reduce steps
n_gpus = 4
angles = np.linspace(0, 2 * np.pi, n_gpus, endpoint=False) - np.pi / 2
radius = 1.0
positions = [(radius * np.cos(a), radius * np.sin(a)) for a in angles]

ax = axes[0]
for i, (x, y) in enumerate(positions):
    circle = plt.Circle((x, y), 0.15, color=f"C{i}", alpha=0.8)
    ax.add_patch(circle)
    ax.text(x, y, f"GPU {i}", ha="center", va="center", fontweight="bold", fontsize=9, color="white")

# Draw ring arrows
for i in range(n_gpus):
    j = (i + 1) % n_gpus
    dx = positions[j][0] - positions[i][0]
    dy = positions[j][1] - positions[i][1]
    length = np.sqrt(dx**2 + dy**2)
    # Shorten arrows to not overlap with circles
    shrink = 0.2 / length
    ax.annotate(
        "", xy=(positions[j][0] - dx * shrink, positions[j][1] - dy * shrink),
        xytext=(positions[i][0] + dx * shrink, positions[i][1] + dy * shrink),
        arrowprops=dict(arrowstyle="->", color="gray", lw=2)
    )

ax.set_xlim(-1.8, 1.8)
ax.set_ylim(-1.8, 1.8)
ax.set_aspect("equal")
ax.set_title("Ring All-Reduce Topology", fontsize=12, fontweight="bold")
ax.axis("off")

# Right: Communication scaling
ax = axes[1]
gpu_counts = np.arange(1, 65)
# All-reduce: 2(N-1)/N * gradient_size (approaches 2x)
comm_per_gpu = 2 * (gpu_counts - 1) / gpu_counts
# Naive broadcast: N-1 copies of gradient_size
naive_comm = gpu_counts - 1

ax.plot(gpu_counts, comm_per_gpu, label="Ring All-Reduce", linewidth=2)
ax.plot(gpu_counts, naive_comm, label="Naive Broadcast", linewidth=2, linestyle="--")
ax.set_xlabel("Number of GPUs", fontsize=11)
ax.set_ylabel("Communication Volume\n(x gradient size per GPU)", fontsize=11)
ax.set_title("Communication Scaling", fontsize=12, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 20)

plt.tight_layout()
plt.show()

## 2. DeepSpeed ZeRO — Stages 1, 2, 3

While DDP is efficient for communication, it wastes memory by replicating everything on every GPU. DeepSpeed's Zero Redundancy Optimizer (ZeRO) addresses this by partitioning the training state across GPUs instead of replicating it. ZeRO comes in three stages, each partitioning more aggressively.

To understand ZeRO, consider what consumes memory during training with AdamW in fp32: the model parameters (4 bytes each), gradients (4 bytes each), and optimizer states (8 bytes each — the first and second moments). For a model with P parameters, the total memory per GPU under DDP is 16P bytes. ZeRO reduces this by partitioning across N GPUs.

**ZeRO Stage 1** partitions optimizer states. Each GPU stores only 1/N of the optimizer states. After all-reduce to get the averaged gradients, each GPU updates only its shard of the parameters using its shard of the optimizer state, then broadcasts the updated parameters. Memory per GPU: 4P (params) + 4P (gradients) + 8P/N (optimizer) = 8P + 8P/N.

**ZeRO Stage 2** partitions both optimizer states and gradients. Gradients are reduced to the GPU that owns the corresponding optimizer shard (reduce-scatter instead of all-reduce). Memory per GPU: 4P (params) + 4P/N (gradients) + 8P/N (optimizer) = 4P + 12P/N.

**ZeRO Stage 3** partitions everything: parameters, gradients, and optimizer states. Parameters are gathered on-demand for each layer during forward and backward passes. Memory per GPU: 4P/N + 4P/N + 8P/N = 16P/N. This means memory scales linearly with the number of GPUs, enabling training of arbitrarily large models.

In [ ]:
def compute_memory_per_gpu(num_params_billions: float, num_gpus: int, stage: int) -> dict:
    """
    Compute memory requirements per GPU for each ZeRO stage.
    
    Assumes fp32 training with AdamW optimizer.
    - Parameters: 4 bytes each
    - Gradients: 4 bytes each
    - Optimizer (AdamW): 8 bytes each (2 momentum buffers)
    
    Returns memory in GB.
    """
    P = num_params_billions * 1e9  # Total parameters
    N = num_gpus
    bytes_per_gb = 1024 ** 3
    
    if stage == 0:  # DDP (no ZeRO)
        params_mem = 4 * P
        grad_mem = 4 * P
        opt_mem = 8 * P
    elif stage == 1:  # Partition optimizer states
        params_mem = 4 * P
        grad_mem = 4 * P
        opt_mem = 8 * P / N
    elif stage == 2:  # Partition optimizer + gradients
        params_mem = 4 * P
        grad_mem = 4 * P / N
        opt_mem = 8 * P / N
    elif stage == 3:  # Partition everything
        params_mem = 4 * P / N
        grad_mem = 4 * P / N
        opt_mem = 8 * P / N
    else:
        raise ValueError(f"Invalid stage: {stage}")
    
    total = params_mem + grad_mem + opt_mem
    
    return {
        "params_gb": params_mem / bytes_per_gb,
        "grads_gb": grad_mem / bytes_per_gb,
        "optimizer_gb": opt_mem / bytes_per_gb,
        "total_gb": total / bytes_per_gb,
    }


# Visualize memory savings for a 7B model
model_size = 7.0  # billion parameters
gpu_counts = [1, 2, 4, 8, 16, 32, 64]
stages = [0, 1, 2, 3]
stage_names = ["DDP (no ZeRO)", "ZeRO-1", "ZeRO-2", "ZeRO-3"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Memory per GPU for different stages
ax = axes[0]
for stage, name in zip(stages, stage_names):
    memories = [compute_memory_per_gpu(model_size, n, stage)["total_gb"] for n in gpu_counts]
    ax.plot(gpu_counts, memories, marker="o", linewidth=2, label=name)

ax.axhline(y=80, color="red", linestyle="--", alpha=0.5, label="A100 80GB")
ax.axhline(y=40, color="orange", linestyle="--", alpha=0.5, label="A100 40GB")
ax.set_xlabel("Number of GPUs", fontsize=11)
ax.set_ylabel("Memory per GPU (GB)", fontsize=11)
ax.set_title(f"Memory per GPU — {model_size:.0f}B Model (fp32)", fontsize=12, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_yscale("log")
ax.set_xscale("log", base=2)
ax.set_xticks(gpu_counts)
ax.set_xticklabels(gpu_counts)

# Right: Memory breakdown for 8 GPUs
ax = axes[1]
x = np.arange(len(stages))
width = 0.25

n_gpus = 8
params = [compute_memory_per_gpu(model_size, n_gpus, s)["params_gb"] for s in stages]
grads = [compute_memory_per_gpu(model_size, n_gpus, s)["grads_gb"] for s in stages]
opts = [compute_memory_per_gpu(model_size, n_gpus, s)["optimizer_gb"] for s in stages]

ax.bar(x, params, width, label="Parameters", color="C0")
ax.bar(x, grads, width, bottom=params, label="Gradients", color="C1")
ax.bar(x, opts, width, bottom=[p + g for p, g in zip(params, grads)], label="Optimizer", color="C2")

ax.set_xlabel("ZeRO Stage", fontsize=11)
ax.set_ylabel("Memory per GPU (GB)", fontsize=11)
ax.set_title(f"Memory Breakdown — {model_size:.0f}B Model, {n_gpus} GPUs", fontsize=12, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(stage_names, rotation=15)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

# Print exact numbers
print(f"\n{'Stage':<20} {'Memory/GPU (GB)':>15}")
print("-" * 40)
for stage, name in zip(stages, stage_names):
    mem = compute_memory_per_gpu(model_size, 8, stage)
    print(f"{name:<20} {mem['total_gb']:>15.1f}")

In [ ]:
def create_deepspeed_config(stage: int = 2) -> dict:
    """
    Create a DeepSpeed configuration for the given ZeRO stage.
    This JSON config is passed to deepspeed.initialize().
    """
    config = {
        "train_batch_size": 256,           # Global batch size across all GPUs
        "train_micro_batch_size_per_gpu": 8,  # Micro-batch per GPU
        "gradient_accumulation_steps": 4,  # 256 / (8 * 8 GPUs) = 4
        
        "optimizer": {
            "type": "AdamW",
            "params": {
                "lr": 3e-4,
                "betas": [0.9, 0.95],
                "eps": 1e-8,
                "weight_decay": 0.1
            }
        },
        
        "scheduler": {
            "type": "WarmupCosineDecayLR",
            "params": {
                "warmup_min_lr": 0,
                "warmup_max_lr": 3e-4,
                "warmup_num_steps": 2000,
                "total_num_steps": 100000
            }
        },
        
        "gradient_clipping": 1.0,
        
        "fp16": {
            "enabled": False  # We'll use bf16 instead
        },
        
        "bf16": {
            "enabled": True
        },
    }
    
    # ZeRO configuration
    if stage == 1:
        config["zero_optimization"] = {
            "stage": 1,
            "reduce_bucket_size": 5e8,
        }
    elif stage == 2:
        config["zero_optimization"] = {
            "stage": 2,
            "allgather_partitions": True,
            "allgather_bucket_size": 5e8,
            "reduce_scatter": True,
            "reduce_bucket_size": 5e8,
            "overlap_comm": True,  # Overlap communication with computation
            "contiguous_gradients": True,
        }
    elif stage == 3:
        config["zero_optimization"] = {
            "stage": 3,
            "allgather_partitions": True,
            "allgather_bucket_size": 5e8,
            "reduce_scatter": True,
            "reduce_bucket_size": 5e8,
            "overlap_comm": True,
            "contiguous_gradients": True,
            "stage3_prefetch_bucket_size": 5e8,
            "stage3_param_persistence_threshold": 1e6,
            "stage3_max_live_parameters": 1e9,
            "stage3_max_reuse_distance": 1e9,
            "stage3_gather_16bit_weights_on_model_save": True,
        }
    
    return config


# Print configs for each stage
for stage in [1, 2, 3]:
    config = create_deepspeed_config(stage)
    print(f"\n=== ZeRO Stage {stage} Config ===")
    print(json.dumps(config["zero_optimization"], indent=2))

### DeepSpeed Training Script

Using DeepSpeed requires minimal changes to your training code. The main differences from DDP are: (1) you use `deepspeed.initialize()` instead of wrapping with DDP, (2) DeepSpeed manages the optimizer and scheduler, and (3) you call `model_engine.step()` instead of separate optimizer and scaler steps. DeepSpeed handles gradient accumulation, mixed precision, and gradient clipping internally.

In [ ]:
def write_deepspeed_training_script():
    script = '''
"""DeepSpeed ZeRO training script.

Launch with:
    deepspeed --num_gpus=NUM_GPUS train_deepspeed.py --deepspeed_config ds_config.json
"""
import torch
import torch.nn as nn
import deepspeed
import argparse

# ... (model and dataset definitions same as DDP script) ...

def train():
    parser = argparse.ArgumentParser()
    parser.add_argument("--local_rank", type=int, default=-1)
    # DeepSpeed adds its own arguments
    parser = deepspeed.add_config_arguments(parser)
    args = parser.parse_args()

    # Create model (NOT on GPU yet — DeepSpeed handles placement)
    model = SimpleLM()

    # Initialize DeepSpeed — this replaces DDP wrapping, optimizer creation, etc.
    model_engine, optimizer, _, scheduler = deepspeed.initialize(
        args=args,
        model=model,
        model_parameters=model.parameters(),
    )

    # Training loop — almost identical to single-GPU code
    dataset = RandomTokenDataset(num_samples=10000, seq_len=256, vocab_size=10000)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=8)

    for epoch in range(2):
        for step, batch in enumerate(dataloader):
            batch = batch.to(model_engine.device)
            logits = model_engine(batch)
            loss = nn.functional.cross_entropy(
                logits[:, :-1].reshape(-1, logits.size(-1)),
                batch[:, 1:].reshape(-1)
            )

            # DeepSpeed handles backward, gradient accumulation, and optimizer step
            model_engine.backward(loss)
            model_engine.step()

            if model_engine.global_rank == 0 and step % 50 == 0:
                print(f"Epoch {epoch} Step {step} Loss: {loss.item():.4f}")

    # Save with DeepSpeed
    model_engine.save_checkpoint("checkpoints/")

if __name__ == "__main__":
    train()
'''
    return script

ds_script = write_deepspeed_training_script()
print("DeepSpeed Training Script:")
print("=" * 60)
print(ds_script)

## 3. FSDP (Fully Sharded Data Parallel) — PyTorch Native

Fully Sharded Data Parallel (FSDP) is PyTorch's native implementation of ZeRO-3 style parameter sharding. It was introduced to provide the same memory efficiency as DeepSpeed ZeRO-3 without requiring a third-party library. FSDP shards model parameters, gradients, and optimizer states across all GPUs, gathering them on-demand for computation.

FSDP works by wrapping individual layers or groups of layers with an FSDP wrapper. Each wrapped unit is handled as an independent sharding unit: its parameters are sharded across all GPUs and gathered only when needed for computation. This is managed automatically — during forward pass, FSDP gathers parameters for the current layer, computes, then discards the full parameters and keeps only the shard. The same happens in reverse during the backward pass.

The wrapping policy is crucial for performance. If you wrap too finely (every single layer), the overhead of all-gather operations becomes significant. If you wrap too coarsely (the entire model), you don't get memory savings during the forward pass. The standard approach is to wrap each transformer block as a unit, which balances memory savings with communication efficiency.

FSDP in PyTorch 2.x has matured significantly and is now the recommended approach for distributed training within the PyTorch ecosystem. It integrates well with PyTorch's compile, activation checkpointing, and mixed precision features.

In [ ]:
def write_fsdp_training_script():
    script = '''
"""FSDP (Fully Sharded Data Parallel) training script.

Launch with:
    torchrun --nproc_per_node=NUM_GPUS train_fsdp.py
"""
import os
import torch
import torch.nn as nn
import torch.distributed as dist
from torch.distributed.fsdp import (
    FullyShardedDataParallel as FSDP,
    MixedPrecision,
    ShardingStrategy,
)
from torch.distributed.fsdp.wrap import (
    transformer_auto_wrap_policy,
)
from torch.utils.data import DataLoader, DistributedSampler
import functools


def setup():
    dist.init_process_group(backend="nccl")
    local_rank = int(os.environ["LOCAL_RANK"])
    torch.cuda.set_device(local_rank)
    return local_rank


# Model definition (same as before)
class TransformerBlock(nn.Module):
    """Individual transformer block — FSDP wraps at this level."""
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Linear(d_model * 4, d_model),
        )
    
    def forward(self, x, mask=None):
        h = self.ln1(x)
        h, _ = self.attn(h, h, h, attn_mask=mask, is_causal=True)
        x = x + h
        x = x + self.ffn(self.ln2(x))
        return x


class TransformerLM(nn.Module):
    def __init__(self, vocab_size=10000, d_model=256, n_heads=4, n_layers=8, max_seq_len=512):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads) for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
    
    def forward(self, input_ids):
        B, T = input_ids.shape
        pos = torch.arange(T, device=input_ids.device).unsqueeze(0)
        x = self.tok_emb(input_ids) + self.pos_emb(pos)
        mask = nn.Transformer.generate_square_subsequent_mask(T, device=input_ids.device)
        for block in self.blocks:
            x = block(x, mask)
        x = self.ln_f(x)
        return self.lm_head(x)


def train():
    local_rank = setup()
    rank = dist.get_rank()
    device = torch.device(f"cuda:{local_rank}")

    # Create model on CPU first (FSDP will shard and move to GPU)
    model = TransformerLM()

    # Mixed precision policy
    mp_policy = MixedPrecision(
        param_dtype=torch.bfloat16,    # Parameters in bf16
        reduce_dtype=torch.bfloat16,   # Gradient reduction in bf16
        buffer_dtype=torch.bfloat16,   # Buffers in bf16
    )

    # Auto-wrap policy: wrap each TransformerBlock
    wrap_policy = functools.partial(
        transformer_auto_wrap_policy,
        transformer_layer_cls={TransformerBlock},
    )

    # Wrap model with FSDP
    model = FSDP(
        model,
        auto_wrap_policy=wrap_policy,
        mixed_precision=mp_policy,
        sharding_strategy=ShardingStrategy.FULL_SHARD,  # ZeRO-3 equivalent
        device_id=local_rank,
        limit_all_gathers=True,  # Limit concurrent all-gathers for memory
    )

    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)

    # Training loop
    dataset = RandomTokenDataset(10000, 256, 10000)
    sampler = DistributedSampler(dataset)
    dataloader = DataLoader(dataset, batch_size=8, sampler=sampler)

    for epoch in range(2):
        sampler.set_epoch(epoch)
        for step, batch in enumerate(dataloader):
            batch = batch.to(device)
            logits = model(batch)
            loss = nn.functional.cross_entropy(
                logits[:, :-1].reshape(-1, logits.size(-1)),
                batch[:, 1:].reshape(-1)
            )
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()

            if rank == 0 and step % 50 == 0:
                print(f"Epoch {epoch} Step {step} Loss: {loss.item():.4f}")

    dist.destroy_process_group()

if __name__ == "__main__":
    train()
'''
    return script

fsdp_script = write_fsdp_training_script()
print("FSDP Training Script:")
print("=" * 60)
print(fsdp_script)

### FSDP vs DeepSpeed ZeRO-3: Comparison

| Feature | FSDP | DeepSpeed ZeRO-3 |
|---------|------|------------------|
| **Ecosystem** | Native PyTorch | Third-party library |
| **Sharding** | Per-wrapper granularity | Per-parameter granularity |
| **Mixed Precision** | Built-in MixedPrecision policy | Config-based (fp16/bf16) |
| **torch.compile** | Full support in PyTorch 2.x | Limited support |
| **Activation Checkpointing** | Native integration | Separate API |
| **CPU Offloading** | Supported | Supported (often faster) |
| **NVMe Offloading** | Not native | Supported (ZeRO-Infinity) |
| **Ease of Use** | More boilerplate | Config-driven, less code |
| **Maturity** | Rapidly improving | More battle-tested |

In practice, FSDP is preferred when you want to stay within the PyTorch ecosystem and take advantage of `torch.compile`. DeepSpeed is preferred when you need CPU/NVMe offloading (ZeRO-Infinity), or when you are working with very large models where DeepSpeed's optimizations for communication overlapping are more mature.

## 4. Mixed Precision Training — fp16, bf16, fp8, Loss Scaling

Mixed precision training uses lower-precision floating point formats (fp16, bf16, or fp8) for most computations while keeping a master copy of weights in fp32. This provides two major benefits: (1) reduced memory — half the bytes per parameter during forward/backward, and (2) faster computation — modern GPUs have specialized tensor cores that compute 2-8x faster in lower precision.

The key insight is that neural network training is surprisingly tolerant of low precision. Gradients and activations don't need the full dynamic range of fp32. However, some operations (like loss computation and weight updates) are sensitive to precision and should remain in fp32. This is why it's called "mixed" precision.

**fp16 (float16)** has 5 exponent bits and 10 mantissa bits. Its limited dynamic range (max ~65504) means that gradients can overflow or underflow. Loss scaling addresses this: we multiply the loss by a large factor before backward (to push small gradients into representable range), then divide the gradients by the same factor before the optimizer step.

**bf16 (bfloat16)** has 8 exponent bits (same as fp32) and 7 mantissa bits. The larger exponent range means it rarely overflows, eliminating the need for loss scaling. This makes bf16 much simpler to use. The tradeoff is slightly less precision (7 vs 10 mantissa bits), but in practice this doesn't matter for training. bf16 is now the standard choice for pretraining on modern hardware (A100, H100, TPU).

**fp8 (float8)** is the newest format, with two variants: E4M3 (4 exponent, 3 mantissa) for forward pass and E5M2 (5 exponent, 2 mantissa) for backward pass. fp8 provides another 2x memory and speed improvement but requires careful calibration and is currently best supported on H100 GPUs.

In [ ]:
# Compare numeric formats
formats = {
    "fp32": {"exponent": 8, "mantissa": 23, "total": 32, "max": 3.4e38, "min_subnormal": 1.4e-45},
    "fp16": {"exponent": 5, "mantissa": 10, "total": 16, "max": 65504, "min_subnormal": 5.96e-8},
    "bf16": {"exponent": 8, "mantissa": 7, "total": 16, "max": 3.39e38, "min_subnormal": 9.2e-41},
    "fp8 (E4M3)": {"exponent": 4, "mantissa": 3, "total": 8, "max": 448, "min_subnormal": 1.95e-3},
    "fp8 (E5M2)": {"exponent": 5, "mantissa": 2, "total": 8, "max": 57344, "min_subnormal": 1.53e-5},
}

print(f"{'Format':<14} {'Bits':>5} {'Exp':>4} {'Man':>4} {'Max Value':>12} {'Memory Saving':>15}")
print("-" * 60)
for name, info in formats.items():
    saving = f"{32/info['total']:.0f}x" if info['total'] < 32 else "baseline"
    print(f"{name:<14} {info['total']:>5} {info['exponent']:>4} {info['mantissa']:>4} {info['max']:>12.2e} {saving:>15}")

In [ ]:
# Visualize the representable ranges
fig, ax = plt.subplots(figsize=(12, 4))

# Log-scale representation of ranges
format_ranges = [
    ("fp32", -45, 38, "C0"),
    ("fp16", -8, 4.8, "C1"),
    ("bf16", -41, 38, "C2"),
    ("fp8 E4M3", -2.7, 2.65, "C3"),
    ("fp8 E5M2", -4.8, 4.76, "C4"),
]

for i, (name, log_min, log_max, color) in enumerate(format_ranges):
    ax.barh(i, log_max - log_min, left=log_min, height=0.5, color=color, alpha=0.7, label=name)
    ax.text(log_max + 0.5, i, name, va="center", fontweight="bold", fontsize=10)

ax.set_xlabel("log10(|value|)", fontsize=11)
ax.set_title("Representable Ranges of Floating Point Formats", fontsize=13, fontweight="bold")
ax.set_yticks([])
ax.axvline(x=0, color="gray", linestyle="--", alpha=0.5)
ax.text(0.3, -0.7, "1.0", color="gray", fontsize=9)
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

In [ ]:
class MixedPrecisionTrainer:
    """
    Demonstrates mixed precision training with fp16 and GradScaler.
    
    The GradScaler workflow:
    1. Forward pass in fp16
    2. Scale the loss by a large factor
    3. Backward pass (gradients are in fp16, scaled up)
    4. Unscale gradients back to original range
    5. If gradients contain inf/nan (overflow), skip the step and reduce scale
    6. Otherwise, optimizer step and potentially increase scale
    """
    def __init__(self, model, lr=3e-4, use_bf16=False):
        self.model = model
        self.optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.1)
        
        self.use_bf16 = use_bf16
        self.dtype = torch.bfloat16 if use_bf16 else torch.float16
        
        # GradScaler only needed for fp16 (bf16 doesn't overflow)
        self.scaler = torch.amp.GradScaler(
            enabled=(not use_bf16 and device.type == "cuda")
        )
    
    def train_step(self, input_ids, labels):
        self.optimizer.zero_grad(set_to_none=True)
        
        # Forward in mixed precision
        with torch.amp.autocast(
            device_type=device.type,
            dtype=self.dtype,
            enabled=(device.type == "cuda")
        ):
            logits = self.model(input_ids)
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=-100
            )
        
        # Backward with scaling (no-op for bf16)
        self.scaler.scale(loss).backward()
        
        # Unscale before gradient clipping
        self.scaler.unscale_(self.optimizer)
        grad_norm = torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
        
        # Step (may skip if gradients overflowed)
        self.scaler.step(self.optimizer)
        self.scaler.update()
        
        return loss.item(), grad_norm.item()


# Demonstrate on CPU (mixed precision is a no-op on CPU but the code structure is the same)
demo_model = MiniTransformerLM(vocab_size=5000, d_model=128, n_heads=4, n_layers=2, max_seq_len=128).to(device)
trainer = MixedPrecisionTrainer(demo_model, lr=3e-4, use_bf16=True)

# Simulate a few steps
for step in range(5):
    dummy_input = torch.randint(0, 5000, (4, 64)).to(device)
    loss, grad_norm = trainer.train_step(dummy_input, dummy_input)
    print(f"Step {step}: loss={loss:.4f}, grad_norm={grad_norm:.4f}")

## 5. Gradient Accumulation — Simulating Larger Batches

Gradient accumulation is a technique for training with effectively larger batch sizes than what fits in GPU memory. Instead of performing an optimizer step after every forward-backward pass, we accumulate gradients over multiple micro-batches and only update the weights after accumulating the desired number of steps.

This is mathematically equivalent to using a larger batch size (up to minor differences from batch normalization, which is rarely used in transformers). If you use gradient accumulation with K micro-batches of size B, the effective batch size is K * B. The gradients are simply summed (or averaged) across micro-batches before the optimizer step.

There is an important subtlety with loss scaling: you must divide the loss (or the gradients) by the number of accumulation steps to get the correct average gradient. If you forget this, you're effectively multiplying the learning rate by the accumulation factor, which can cause instability.

In distributed training, gradient accumulation interacts with communication: you should NOT synchronize gradients on intermediate micro-steps. DDP provides a `no_sync()` context manager for this purpose. Only the final micro-step should trigger the all-reduce.

In [ ]:
def train_with_gradient_accumulation(
    model: nn.Module,
    dataloader: DataLoader,
    optimizer: torch.optim.Optimizer,
    num_steps: int = 50,
    grad_accum_steps: int = 4,
    max_grad_norm: float = 1.0,
) -> list:
    """
    Training loop with proper gradient accumulation.
    
    Key points:
    - Loss is divided by grad_accum_steps for correct averaging
    - Optimizer step only happens every grad_accum_steps micro-batches
    - Gradient clipping happens before the optimizer step
    """
    model.train()
    losses = []
    data_iter = iter(dataloader)
    global_step = 0
    micro_step = 0
    accumulated_loss = 0.0
    
    while global_step < num_steps:
        try:
            batch = next(data_iter)
        except StopIteration:
            data_iter = iter(dataloader)
            batch = next(data_iter)
        
        input_ids = batch.to(device)
        
        # Forward
        logits = model(input_ids)
        loss = F.cross_entropy(
            logits[:, :-1].reshape(-1, logits.size(-1)),
            input_ids[:, 1:].reshape(-1)
        )
        
        # CRITICAL: divide by accumulation steps for correct gradient averaging
        scaled_loss = loss / grad_accum_steps
        scaled_loss.backward()
        
        accumulated_loss += loss.item()
        micro_step += 1
        
        # Optimizer step only every grad_accum_steps
        if micro_step % grad_accum_steps == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            
            avg_loss = accumulated_loss / grad_accum_steps
            losses.append(avg_loss)
            accumulated_loss = 0.0
            global_step += 1
    
    return losses


# Compare different effective batch sizes via gradient accumulation
fig, ax = plt.subplots(figsize=(10, 5))

configs = [
    {"micro_batch": 4, "accum": 1, "label": "Effective BS=4 (no accum)"},
    {"micro_batch": 4, "accum": 4, "label": "Effective BS=16 (accum=4)"},
    {"micro_batch": 4, "accum": 16, "label": "Effective BS=64 (accum=16)"},
]

# Create a dataset
from torch.utils.data import TensorDataset
train_tokens = torch.randint(0, 5000, (2000, 64))
dataset = TensorDataset(train_tokens)

class SimpleDataset(Dataset):
    def __init__(self, data):
        self.data = data
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return self.data[idx]

ds = SimpleDataset(train_tokens)

for config in configs:
    torch.manual_seed(42)
    test_model = MiniTransformerLM(
        vocab_size=5000, d_model=64, n_heads=2, n_layers=1, max_seq_len=64
    ).to(device)
    test_optimizer = torch.optim.AdamW(test_model.parameters(), lr=3e-4)
    loader = DataLoader(ds, batch_size=config["micro_batch"], shuffle=True, drop_last=True)
    
    losses = train_with_gradient_accumulation(
        test_model, loader, test_optimizer,
        num_steps=50, grad_accum_steps=config["accum"]
    )
    
    ax.plot(losses, label=config["label"], linewidth=2)

ax.set_xlabel("Optimizer Step", fontsize=11)
ax.set_ylabel("Loss", fontsize=11)
ax.set_title("Effect of Gradient Accumulation (Different Effective Batch Sizes)", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
def write_ddp_gradient_accumulation_snippet():
    """
    Show how gradient accumulation works with DDP.
    The key is using model.no_sync() on intermediate steps
    to avoid unnecessary gradient synchronization.
    """
    snippet = '''
# Gradient accumulation with DDP — the correct way
# model is wrapped with DistributedDataParallel

grad_accum_steps = 4
optimizer.zero_grad()

for micro_step in range(grad_accum_steps):
    batch = next(data_iter)
    
    # On intermediate steps, skip gradient synchronization
    # This avoids N-1 unnecessary all-reduce operations
    context = model.no_sync if micro_step < grad_accum_steps - 1 else nullcontext
    
    with context():
        logits = model(batch)
        loss = compute_loss(logits, batch) / grad_accum_steps
        loss.backward()  # Gradients accumulate; no all-reduce on no_sync steps

# After the loop, the final backward() triggered the all-reduce
torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
optimizer.step()
optimizer.zero_grad()
'''
    return snippet

print("DDP Gradient Accumulation Pattern:")
print(write_ddp_gradient_accumulation_snippet())

## 6. Data Loading at Scale — Streaming Datasets, Distributed Sharding

At scale, data loading becomes a critical bottleneck. Pretraining datasets can be hundreds of gigabytes to multiple terabytes — far too large to fit in memory. We need streaming approaches that load and process data on the fly, while ensuring that distributed workers get disjoint data shards.

HuggingFace Datasets provides a streaming mode that loads data lazily from disk or remote storage. Instead of downloading the entire dataset, it streams examples one by one (or in small chunks), processes them, and feeds them to the model. This keeps memory usage constant regardless of dataset size.

In distributed training, each GPU must see different data. This requires coordinated sharding: the dataset is logically split into N shards (one per GPU), and each GPU only iterates over its shard. HuggingFace Datasets' streaming mode supports this natively via `.shard(num_shards=world_size, index=rank)`.

In [ ]:
def write_streaming_data_pipeline():
    """
    A production streaming data pipeline for pretraining,
    using HuggingFace datasets in streaming mode.
    """
    code = '''
from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import IterableDataset, DataLoader
import torch
import torch.distributed as dist


class StreamingPretrainDataset(IterableDataset):
    """
    Streaming dataset that tokenizes and packs sequences on-the-fly.
    Handles distributed sharding automatically.
    """
    def __init__(
        self,
        dataset_name: str = "allenai/c4",
        split: str = "train",
        tokenizer_name: str = "meta-llama/Llama-2-7b-hf",
        max_seq_len: int = 2048,
        text_column: str = "text",
    ):
        self.max_seq_len = max_seq_len
        self.text_column = text_column
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
        
        # Load in streaming mode (no download)
        self.dataset = load_dataset(
            dataset_name, split=split, streaming=True
        )
    
    def _shard_for_distributed(self, dataset):
        """Shard dataset for distributed training."""
        if dist.is_initialized():
            world_size = dist.get_world_size()
            rank = dist.get_rank()
            dataset = dataset.shard(num_shards=world_size, index=rank)
        return dataset
    
    def _pack_tokens(self, token_iterator):
        """Pack tokenized documents into fixed-length sequences."""
        buffer = []
        
        for tokens in token_iterator:
            buffer.extend(tokens)
            
            while len(buffer) >= self.max_seq_len:
                yield torch.tensor(buffer[:self.max_seq_len], dtype=torch.long)
                buffer = buffer[self.max_seq_len:]
    
    def __iter__(self):
        dataset = self._shard_for_distributed(self.dataset)
        
        # Tokenize documents as they stream in
        def tokenize_stream():
            for example in dataset:
                text = example[self.text_column]
                tokens = self.tokenizer.encode(text, add_special_tokens=False)
                if tokens:  # Skip empty documents
                    # Add EOS token between documents
                    tokens.append(self.tokenizer.eos_token_id)
                    yield tokens
        
        # Pack into fixed-length sequences
        yield from self._pack_tokens(tokenize_stream())


# Usage:
# dataset = StreamingPretrainDataset(
#     dataset_name="allenai/c4",
#     tokenizer_name="meta-llama/Llama-2-7b-hf",
#     max_seq_len=2048,
# )
# dataloader = DataLoader(dataset, batch_size=8, num_workers=4)
# for batch in dataloader:
#     # batch shape: (8, 2048)
#     ...
'''
    return code

print("Streaming Data Pipeline:")
print(write_streaming_data_pipeline())

In [ ]:
# Simulate streaming data loading to demonstrate the pattern locally

class SimulatedStreamingDataset(torch.utils.data.IterableDataset):
    """
    Simulates a streaming dataset for local demonstration.
    Generates random token sequences that mimic the behavior
    of a real streaming pipeline.
    """
    def __init__(self, vocab_size=5000, max_seq_len=128, num_docs=10000, rank=0, world_size=1):
        self.vocab_size = vocab_size
        self.max_seq_len = max_seq_len
        self.num_docs = num_docs
        self.rank = rank
        self.world_size = world_size
    
    def _generate_documents(self):
        """Simulate document generation with variable lengths."""
        rng = np.random.RandomState(42 + self.rank)
        for i in range(self.rank, self.num_docs, self.world_size):
            doc_len = int(rng.exponential(scale=80)) + 5
            yield list(rng.randint(4, self.vocab_size, size=doc_len))
    
    def _pack_documents(self, doc_iter):
        """Pack documents into fixed-length sequences."""
        buffer = []
        eos_token = 2
        for doc in doc_iter:
            buffer.extend(doc)
            buffer.append(eos_token)
            while len(buffer) >= self.max_seq_len:
                yield torch.tensor(buffer[:self.max_seq_len], dtype=torch.long)
                buffer = buffer[self.max_seq_len:]
    
    def __iter__(self):
        yield from self._pack_documents(self._generate_documents())


# Test streaming data loader
streaming_ds = SimulatedStreamingDataset(vocab_size=5000, max_seq_len=128, num_docs=500)
streaming_loader = DataLoader(streaming_ds, batch_size=8)

# Count batches and tokens
total_batches = 0
total_tokens = 0
for batch in streaming_loader:
    total_batches += 1
    total_tokens += batch.numel()

print(f"Streamed {total_batches} batches")
print(f"Total tokens: {total_tokens:,}")
print(f"Batch shape: {batch.shape}")
print(f"\nSimulated 4-GPU sharding:")
for rank in range(4):
    ds = SimulatedStreamingDataset(vocab_size=5000, max_seq_len=128, num_docs=500, rank=rank, world_size=4)
    count = sum(1 for _ in DataLoader(ds, batch_size=8))
    print(f"  GPU {rank}: {count} batches")

## Summary

In this notebook, we covered the key techniques for scaling pretraining beyond a single GPU:

1. **DDP (Distributed Data Parallel)**: The simplest approach — replicate the model, split the data, synchronize gradients. Works when the model fits on one GPU.

2. **DeepSpeed ZeRO**: Progressively partition optimizer states (Stage 1), gradients (Stage 2), and parameters (Stage 3) across GPUs. Enables training models that don't fit on a single GPU.

3. **FSDP**: PyTorch's native ZeRO-3 implementation. Wraps transformer blocks for automatic parameter sharding with seamless `torch.compile` integration.

4. **Mixed Precision**: Use bf16 (preferred) or fp16 with loss scaling to cut memory by 2x and speed up training with tensor cores.

5. **Gradient Accumulation**: Simulate larger batch sizes without more memory. Critical detail: use `model.no_sync()` in DDP for intermediate steps.

6. **Streaming Data**: Stream and pack data on-the-fly for datasets that don't fit in memory. Shard across GPUs for distributed training.

---

**Next:** [expert.ipynb](./expert.ipynb) — Scaling laws, compute-optimal training, data quality, and training dynamics.